# Enhanced Call of Duty Performance Analysis

This notebook provides a comprehensive analysis of Call of Duty game performance using modular components for improved maintainability and extensibility.

## Features
- Modular design with separate modules for parsing, processing, visualization, and statistics
- Advanced statistical analysis and hypothesis testing
- Interactive visualizations and performance dashboards
- Trend analysis and anomaly detection
- Player comparison and ranking systems

## Analysis Sections
1. Data Loading and Preprocessing
2. Exploratory Data Analysis
3. Performance Metrics Calculation
4. Statistical Analysis and Testing
5. Advanced Visualizations
6. Player Performance Comparison
7. Time Series Analysis
8. Game Mode and Map Analysis
9. Predictive Modeling (Optional)
10. Summary Report Generation

## 1. Import Libraries and Modules

In [3]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import logging

# Configure warnings and logging
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

# Our custom modules
from data_parser import CODDataParser, load_player_data
from data_processor import CODDataProcessor, process_cod_data, add_advanced_features
from visualization import CODVisualizer, create_performance_dashboard
from cod_statistics import CODStatisticalAnalyzer, perform_comprehensive_analysis
from config import *

# Set up matplotlib for inline plots
%matplotlib inline
plt.rcParams['figure.figsize'] = FIGURE_SIZES['medium']
sns.set_style(CHART_STYLE['style'])

print("✅ All modules imported successfully!")
print(f"📊 Supported games: {len(SUPPORTED_GAMES)}")
print(f"👥 Default players: {DEFAULT_PLAYERS}")

✅ All modules imported successfully!
📊 Supported games: 6
👥 Default players: ['Mystyy', 'Glovali', 'Risky', 'Anima']


## 2. Data Loading and Preprocessing

### 📁 Data Setup Instructions

**Important**: This notebook expects HTML files from Activision accounts containing match data.

#### To get your data:
1. Log into your Activision account
2. Navigate to your match history 
3. Save the HTML pages for each player
4. Place files in a `data/` directory in this project

#### Expected file structure:
```
cod_analysis/
├── data/
│   ├── player1-ACTIVISION_ACCOUNT.html
│   ├── player2-ACTIVISION_ACCOUNT.html
│   └── ...
├── enhanced_analysis.ipynb
└── ...
```

#### File mapping:
Update the file paths in `config.py` or modify the `file_mapping` variable below to match your actual files.

**Note**: If no data files are found, the notebook will create sample data for demonstration purposes.

In [5]:
# Initialize the parser and processor
parser = CODDataParser(SUPPORTED_GAMES)
processor = CODDataProcessor()

# Load data using the default file mapping
print("📂 Loading player data files...")
file_mapping = get_default_file_mapping()

# Display file mapping
for file_path, player in file_mapping.items():
    print(f"  {player}: {file_path}")

# Check if data files exist
import os
missing_files = []
for file_path in file_mapping.keys():
    if not os.path.exists(file_path):
        missing_files.append(file_path)

if missing_files:
    print(f"\n⚠️  Missing data files:")
    for file_path in missing_files:
        print(f"  - {file_path}")
    print(f"\n📝 To use this notebook:")
    print(f"1. Create a 'data' directory in the project root")
    print(f"2. Place your Activision HTML files in the data directory")
    print(f"3. Update the file mapping in config.py or below")
    print(f"\n🔧 For now, let's create some sample data for demonstration...")
    
    # Create sample data for demonstration
    sample_data = {
        'Player': ['Glovali', 'Mystyy', 'Risky', 'Anima'] * 25,
        'Match ID': [f'match_{i}' for i in range(100)],
        'Game Name': [' Call of Duty: Black Ops 6'] * 100,
        'UTC Timestamp': pd.date_range(start='2024-01-01', periods=100, freq='D'),
        'Kills': np.random.randint(5, 35, 100),
        'Deaths': np.random.randint(3, 25, 100),
        'Game Type': np.random.choice(['Domination', 'Hardpoint', 'Kill Confirmed'], 100),
        'Match Start Timestamp': pd.date_range(start='2024-01-01', periods=100, freq='D'),
        'Match End Timestamp': pd.date_range(start='2024-01-01 00:30:00', periods=100, freq='D'),
        'Map': np.random.choice(['Nuketown', 'Babylon', 'Derelict'], 100),
        'Match Outcome': np.random.choice(['Victory', 'Defeat'], 100),
        'Skill': np.random.uniform(0.8, 2.5, 100),
        'Score': np.random.randint(1000, 8000, 100),
        'Shots': np.random.randint(50, 200, 100),
        'Hits': np.random.randint(20, 120, 100),
        'Assists': np.random.randint(0, 15, 100),
        'Longest Streak': np.random.randint(0, 12, 100),
        'Headshots': np.random.randint(0, 10, 100),
        'Damage Done': np.random.uniform(1000, 6000, 100),
        'Match XP': np.random.randint(500, 3000, 100)
    }
    
    raw_df = pd.DataFrame(sample_data)
    print(f"\n🎲 Created sample dataset for demonstration")
else:
    # Parse and combine all data
    raw_df = parser.parse_multiple_files(file_mapping)

print(f"\n📊 Raw data loaded: {len(raw_df)} records")
if len(raw_df) > 0:
    print(f"👥 Players found: {raw_df['Player'].unique().tolist()}")
    if 'Game Name' in raw_df.columns:
        print(f"🎮 Games found: {raw_df['Game Name'].unique().tolist()}")
else:
    print("❌ No data loaded. Please check file paths and try again.")

📂 Loading player data files...
  Glovali: data/33833496-ACTIVISION_ACCOUNT.html
  Mystyy: data/33815277-ACTIVISION_ACCOUNT.html
  Risky: data/33810648-ACTIVISION_ACCOUNT.html
  Anima: data/33757681-ACTIVISION_ACCOUNT.html

⚠️  Missing data files:
  - data/33833496-ACTIVISION_ACCOUNT.html
  - data/33815277-ACTIVISION_ACCOUNT.html
  - data/33810648-ACTIVISION_ACCOUNT.html
  - data/33757681-ACTIVISION_ACCOUNT.html

📝 To use this notebook:
1. Create a 'data' directory in the project root
2. Place your Activision HTML files in the data directory
3. Update the file mapping in config.py or below

🔧 For now, let's create some sample data for demonstration...

🎲 Created sample dataset for demonstration

📊 Raw data loaded: 100 records
👥 Players found: ['Glovali', 'Mystyy', 'Risky', 'Anima']
🎮 Games found: [' Call of Duty: Black Ops 6']


In [ ]:
# 🔧 OPTIONAL: Update file mapping if you have your own data files
# Uncomment and modify the lines below to point to your actual HTML files

# Custom file mapping - update these paths to match your files
custom_file_mapping = {
    # "data/your_file1.html": "Player1Name",
    # "data/your_file2.html": "Player2Name",
    # Add more files as needed...
}

# Uncomment the line below to use your custom mapping instead of the default
# file_mapping = custom_file_mapping

print("💡 Tip: If you have your own data files, update the custom_file_mapping above!")

In [6]:
# Process the raw data
print("🔄 Processing and cleaning data...")
processed_df = processor.clean_and_process(raw_df)

# Add advanced features
print("✨ Adding advanced features...")
enhanced_df = add_advanced_features(processed_df)

print(f"\n✅ Data processing complete!")
print(f"📊 Final dataset: {len(enhanced_df)} records")
print(f"📈 Columns available: {len(enhanced_df.columns)}")

# Display basic info about the processed data
enhanced_df.info()

INFO:data_processor:Processing 100 records
INFO:data_processor:Found 1 outliers in Accuracy
INFO:data_processor:Total outliers handled: 1
INFO:data_processor:Processing complete: 100 records


🔄 Processing and cleaning data...
✨ Adding advanced features...

✅ Data processing complete!
📊 Final dataset: 100 records
📈 Columns available: 47
<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 3 to 98
Data columns (total 47 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Player                 100 non-null    object        
 1   Match ID               100 non-null    object        
 2   Game Name              100 non-null    object        
 3   UTC Timestamp          100 non-null    datetime64[ns]
 4   Kills                  100 non-null    Int64         
 5   Deaths                 100 non-null    Int64         
 6   Game Type              100 non-null    object        
 7   Match Start Timestamp  100 non-null    datetime64[ns]
 8   Match End Timestamp    100 non-null    datetime64[ns]
 9   Map                    100 non-null    object        
 10  Match Outcome          100 non-null    obj

## 3. Exploratory Data Analysis

In [ ]:
# Basic dataset overview
print("📋 DATASET OVERVIEW")
print("=" * 50)

# Time range
if 'UTC Timestamp' in enhanced_df.columns:
    start_date = enhanced_df['UTC Timestamp'].min()
    end_date = enhanced_df['UTC Timestamp'].max()
    print(f"📅 Date range: {start_date.date()} to {end_date.date()}")
    print(f"⏱️  Total days: {(end_date - start_date).days}")

# Player statistics
player_counts = enhanced_df['Player'].value_counts()
print(f"\n👥 PLAYER MATCH COUNTS:")
for player, count in player_counts.items():
    print(f"  {player}: {count} matches")

# Game statistics
game_counts = enhanced_df['Game Name'].value_counts()
print(f"\n🎮 GAME DISTRIBUTION:")
for game, count in game_counts.items():
    print(f"  {GAME_ABBREVIATIONS.get(game, game)}: {count} matches")

# Game mode statistics
if 'Game Type' in enhanced_df.columns:
    mode_counts = enhanced_df['Game Type'].value_counts().head(10)
    print(f"\n🎯 TOP GAME MODES:")
    for mode, count in mode_counts.items():
        print(f"  {mode}: {count} matches")

In [ ]:
# Display key performance metrics summary
key_metrics = [col for col in CORE_METRICS if col in enhanced_df.columns]

if key_metrics:
    print("📊 PERFORMANCE METRICS SUMMARY")
    print("=" * 50)
    
    summary_stats = enhanced_df.groupby('Player')[key_metrics].agg(['mean', 'std', 'count']).round(2)
    display(summary_stats)
else:
    print("⚠️ No key metrics found in the dataset")

## 4. Performance Metrics Visualization

In [ ]:
# Initialize visualizer
visualizer = CODVisualizer(figsize=FIGURE_SIZES['large'])

# Create performance over time plots for key metrics
metrics_to_plot = ['KD_Ratio', 'SPM', 'Skill', 'Accuracy']
available_metrics = [m for m in metrics_to_plot if m in enhanced_df.columns]

print(f"📈 Creating performance trend plots for: {available_metrics}")

for metric in available_metrics:
    fig = visualizer.plot_performance_over_time(
        enhanced_df, 
        metric, 
        show_trend=True
    )
    plt.show()
    plt.close()

In [ ]:
# Player comparison radar chart
comparison_metrics = [m for m in ['KD_Ratio', 'SPM', 'Accuracy', 'Score'] if m in enhanced_df.columns]

if len(comparison_metrics) >= 3:
    print(f"🕸️ Creating player comparison radar chart for: {comparison_metrics}")
    fig = visualizer.plot_player_comparison(enhanced_df, comparison_metrics)
    plt.show()
    plt.close()
else:
    print("⚠️ Not enough metrics available for radar chart")

## 5. Statistical Analysis

In [ ]:
# Initialize statistical analyzer
analyzer = CODStatisticalAnalyzer(confidence_level=0.95)

# Perform comprehensive analysis
print("🔬 Performing comprehensive statistical analysis...")
analysis_results = perform_comprehensive_analysis(enhanced_df)

# Display dataset summary
print("\n📋 DATASET SUMMARY")
print("=" * 50)
dataset_summary = analysis_results['dataset_summary']
print(f"Total matches: {dataset_summary['total_matches']}")
print(f"Players: {', '.join(dataset_summary['players'])}")
print(f"Games: {len(dataset_summary['games'])}")
if dataset_summary['date_range']['start']:
    print(f"Date range: {dataset_summary['date_range']['start']} to {dataset_summary['date_range']['end']}")

In [ ]:
# Player performance comparisons
if 'player_comparisons' in analysis_results:
    print("\n👥 PLAYER PERFORMANCE COMPARISONS")
    print("=" * 50)
    
    for metric, comparison in analysis_results['player_comparisons'].items():
        if 'error' not in comparison:
            print(f"\n📊 {get_metric_display_name(metric)}:")
            
            # Display means
            print("  Player averages:")
            for player, mean_val in comparison['means'].items():
                std_val = comparison['stds'][player]
                print(f"    {player}: {mean_val:.3f} (±{std_val:.3f})")
            
            # Display significant differences
            significant_tests = [test for test, result in comparison['tests'].items() if result['significant']]
            if significant_tests:
                print(f"  🔍 Significant differences found in: {', '.join(significant_tests)}")
            else:
                print("  📊 No statistically significant differences found")

In [ ]:
# Correlation analysis
if 'correlation_analysis' in analysis_results:
    correlation_data = analysis_results['correlation_analysis']
    
    if 'strong_correlations' in correlation_data and correlation_data['strong_correlations']:
        print("\n🔗 STRONG CORRELATIONS FOUND")
        print("=" * 50)
        
        for corr in correlation_data['strong_correlations']:
            print(f"📈 {corr['variables']}: r = {corr['pearson_r']:.3f} ({corr['strength']})")
    else:
        print("\n🔗 No strong correlations found (r > 0.5)")

## 6. Advanced Visualizations

In [ ]:
# Correlation matrix heatmap
metrics_for_correlation = [m for m in CORE_METRICS if m in enhanced_df.columns]

if len(metrics_for_correlation) >= 3:
    print(f"🔥 Creating correlation matrix for: {metrics_for_correlation}")
    fig = visualizer.plot_correlation_matrix(enhanced_df, metrics_for_correlation)
    plt.show()
    plt.close()
else:
    print("⚠️ Not enough metrics for correlation matrix")

In [ ]:
# Performance distribution plots
key_metric = 'KD_Ratio' if 'KD_Ratio' in enhanced_df.columns else CORE_METRICS[0]

if key_metric in enhanced_df.columns:
    print(f"📊 Creating distribution plots for {get_metric_display_name(key_metric)}")
    fig = visualizer.plot_performance_distribution(enhanced_df, key_metric, 'Player')
    plt.show()
    plt.close()
else:
    print("⚠️ No suitable metrics for distribution plots")

In [ ]:
# Game mode performance comparison
if 'Game Type' in enhanced_df.columns and key_metric in enhanced_df.columns:
    print(f"🎯 Creating game mode performance comparison")
    fig = visualizer.plot_game_mode_performance(enhanced_df, key_metric)
    plt.show()
    plt.close()
else:
    print("⚠️ Game Type or performance metric not available")

In [ ]:
# Map performance comparison (if map data is available)
if 'Map' in enhanced_df.columns and key_metric in enhanced_df.columns:
    print(f"🗺️ Creating map performance comparison")
    fig = visualizer.plot_map_performance(enhanced_df, key_metric, top_n=10)
    plt.show()
    plt.close()
else:
    print("⚠️ Map or performance metric not available")

## 7. Win Rate Analysis

In [ ]:
# Win rate analysis
if 'Match Outcome' in enhanced_df.columns:
    print("🏆 Creating win rate analysis dashboard")
    fig = visualizer.plot_win_rate_analysis(enhanced_df)
    plt.show()
    plt.close()
else:
    print("⚠️ Match outcome data not available for win rate analysis")

## 8. Individual Player Analysis

In [ ]:
# Analyze trends for each player
print("🔍 INDIVIDUAL PLAYER TREND ANALYSIS")
print("=" * 50)

for player in DEFAULT_PLAYERS:
    if player in enhanced_df['Player'].values:
        print(f"\n👤 Analyzing {player}...")
        
        # Analyze KD ratio trends
        if 'KD_Ratio' in enhanced_df.columns:
            trend_analysis = analyzer.analyze_performance_trends(
                enhanced_df, player, 'KD_Ratio', window=10
            )
            
            if 'error' not in trend_analysis:
                trend_info = trend_analysis['trend_analysis']
                stats_info = trend_analysis['statistics']
                
                print(f"  📈 K/D Trend: {trend_info['trend_direction']} (slope: {trend_info['slope']:.4f})")
                print(f"  📊 Overall K/D: {stats_info['overall_mean']:.3f} (±{stats_info['overall_std']:.3f})")
                print(f"  🎯 Recent K/D: {stats_info['recent_mean']:.3f}")
                print(f"  🏆 Best: {stats_info['best_performance']:.3f}, Worst: {stats_info['worst_performance']:.3f}")
            else:
                print(f"  ⚠️ {trend_analysis['error']}")
        
        # Detect anomalies
        if 'KD_Ratio' in enhanced_df.columns:
            anomaly_analysis = analyzer.detect_performance_anomalies(
                enhanced_df, player, 'KD_Ratio', threshold=2.0
            )
            
            if 'error' not in anomaly_analysis:
                anomaly_rate = anomaly_analysis['anomaly_rate']
                print(f"  🚨 Anomaly rate: {anomaly_rate:.1f}% ({anomaly_analysis['anomalies_found']} anomalies)")
            else:
                print(f"  ⚠️ {anomaly_analysis['error']}")
    else:
        print(f"\n👤 {player}: No data found")

## 9. Game-Specific Analysis

In [ ]:
# Analyze performance by game
print("🎮 GAME-SPECIFIC PERFORMANCE ANALYSIS")
print("=" * 50)

for game in enhanced_df['Game Name'].unique():
    game_data = enhanced_df[enhanced_df['Game Name'] == game]
    game_abbrev = GAME_ABBREVIATIONS.get(game, game)
    
    print(f"\n🎯 {game_abbrev} ({len(game_data)} matches):")
    
    # Player performance in this game
    if 'KD_Ratio' in game_data.columns:
        player_performance = game_data.groupby('Player')['KD_Ratio'].agg(['mean', 'count']).round(3)
        print("  Player K/D averages:")
        for player, stats in player_performance.iterrows():
            if stats['count'] >= 5:  # Only show players with sufficient data
                print(f"    {player}: {stats['mean']:.3f} ({stats['count']} matches)")
    
    # Game impact analysis
    if len(enhanced_df['Game Name'].unique()) > 1 and 'KD_Ratio' in enhanced_df.columns:
        game_impact = analyzer.analyze_game_impact(enhanced_df, 'KD_Ratio')
        if 'error' not in game_impact and game_impact['kruskal_wallis']['significant']:
            print(f"  📊 Significant performance differences across games (p < 0.05)")

## 10. Player Rankings and Summary

In [ ]:
# Calculate comprehensive player rankings
ranking_metrics = [m for m in ['KD_Ratio', 'SPM', 'Skill', 'Accuracy'] if m in enhanced_df.columns]

if len(ranking_metrics) >= 2:
    print("🏆 PLAYER RANKINGS")
    print("=" * 50)
    
    rankings = analyzer.calculate_player_rankings(enhanced_df, ranking_metrics)
    
    print(f"Based on metrics: {', '.join([get_metric_display_name(m) for m in ranking_metrics])}")
    print("\nOverall Rankings:")
    
    for i, (player, row) in enumerate(rankings.iterrows(), 1):
        print(f"  {i}. {player} (Score: {row['overall_rank']:.2f})")
        
        # Show individual metric ranks
        for metric in ranking_metrics:
            rank_col = f'{metric}_rank'
            mean_col = f'{metric}_mean'
            if rank_col in row and mean_col in row:
                print(f"     {get_metric_display_name(metric)}: #{int(row[rank_col])} ({row[mean_col]:.3f})")
        print()
else:
    print("⚠️ Not enough metrics available for comprehensive rankings")

## 11. Key Insights and Recommendations

In [ ]:
# Generate key insights
print("💡 KEY INSIGHTS AND RECOMMENDATIONS")
print("=" * 60)

insights = []

# Data quality insights
total_matches = len(enhanced_df)
players_count = len(enhanced_df['Player'].unique())
games_count = len(enhanced_df['Game Name'].unique())

insights.append(f"📊 Dataset contains {total_matches} matches across {games_count} games for {players_count} players")

# Performance insights
if 'KD_Ratio' in enhanced_df.columns:
    avg_kd = enhanced_df['KD_Ratio'].mean()
    best_player = enhanced_df.groupby('Player')['KD_Ratio'].mean().idxmax()
    best_kd = enhanced_df.groupby('Player')['KD_Ratio'].mean().max()
    
    insights.append(f"🎯 Average K/D ratio across all players: {avg_kd:.3f}")
    insights.append(f"🏆 Best average K/D: {best_player} ({best_kd:.3f})")

# Game distribution insights
most_played_game = enhanced_df['Game Name'].value_counts().index[0]
most_played_count = enhanced_df['Game Name'].value_counts().iloc[0]
insights.append(f"🎮 Most played game: {GAME_ABBREVIATIONS.get(most_played_game, most_played_game)} ({most_played_count} matches)")

# Activity insights
if 'UTC Timestamp' in enhanced_df.columns:
    date_range = (enhanced_df['UTC Timestamp'].max() - enhanced_df['UTC Timestamp'].min()).days
    matches_per_day = total_matches / max(date_range, 1)
    insights.append(f"📅 Average matches per day: {matches_per_day:.1f}")

# Display insights
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")

print("\n🎯 RECOMMENDATIONS:")
recommendations = [
    "Consider analyzing performance patterns by time of day or day of week",
    "Investigate correlation between win rate and individual performance metrics", 
    "Analyze the impact of different weapon loadouts on performance (if data available)",
    "Track improvement over time to identify effective practice strategies",
    "Compare performance across different game modes to identify strengths"
]

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

## 12. Export Results (Optional)

In [ ]:
# Export processed data and analysis results
import os
from datetime import datetime

# Create output directories
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f"{output_dir}/data", exist_ok=True)
os.makedirs(f"{output_dir}/reports", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    # Export processed data
    output_file = f"{output_dir}/data/processed_cod_data_{timestamp}.csv"
    enhanced_df.to_csv(output_file, index=False)
    print(f"✅ Processed data exported to: {output_file}")
    
    # Export summary statistics
    if key_metrics:
        summary_file = f"{output_dir}/reports/player_summary_{timestamp}.csv"
        player_summary = enhanced_df.groupby('Player')[key_metrics].agg(['mean', 'std', 'count']).round(3)
        player_summary.to_csv(summary_file)
        print(f"✅ Player summary exported to: {summary_file}")
    
    print(f"\n📁 All outputs saved in: {os.path.abspath(output_dir)}")
    
except Exception as e:
    print(f"⚠️ Error exporting data: {e}")
    print("You may need to create the output directories manually")

## Summary

This enhanced analysis notebook provides a comprehensive view of Call of Duty performance data with:

✅ **Modular Design**: Separate modules for parsing, processing, visualization, and statistics  
✅ **Advanced Statistics**: Hypothesis testing, correlation analysis, and trend detection  
✅ **Rich Visualizations**: Interactive plots, radar charts, and performance dashboards  
✅ **Player Comparisons**: Statistical significance testing and ranking systems  
✅ **Anomaly Detection**: Identify unusual performances and outliers  
✅ **Export Capabilities**: Save processed data and analysis results  

### Next Steps
- Add predictive modeling to forecast future performance
- Implement real-time data updates and monitoring
- Create interactive dashboards using Plotly/Bokeh
- Add more sophisticated machine learning analysis
- Integrate weapon and loadout analysis if data becomes available